# Spark Structured Streaming — 实时消费 Kafka 数据

**流程：** Kafka `tlc-trips` topic → Spark Structured Streaming → 实时聚合结果

运行前确认：
1. Kafka broker 正在运行（终端 1）
2. topic `tlc-trips` 已创建
3. notebook 02 的 Producer 准备好随时发送数据

## 1. 初始化 SparkSession（带 Kafka 连接器）

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("TLC-Streaming")
    .config("spark.sql.shuffle.partitions", "4")
    # Kafka 连接器，Spark 会自动下载这个 jar 包
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("SparkSession ready.")

26/09/28 19:44:47 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/09/28 19:44:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/Users/lujiayi/anaconda3/envs/tlc-spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lujiayi/.ivy2/cache
The jars for the packages stored in: /Users/lujiayi/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-154c12e3-b4e6-4d5c-a477-46c3cda52531;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central


	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central


:: resolution report :: resolve 668ms :: artifacts dl 21ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from local-m2-cache in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.slf4j#slf4j-api;2.0.7 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|

26/09/28 19:44:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1
SparkSession ready.


## 2. 从 Kafka 读取数据流

In [2]:
# readStream：持续监听，不是一次性读取
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "tlc-trips")
    .option("startingOffsets", "earliest")  # 从头开始读，包含之前发的 500 条
    .load()
)

print("Stream schema:")
raw_stream.printSchema()

Stream schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## 3. 解析 JSON 消息

In [3]:
# 定义消息的 schema（和 Producer 发送的字段对应）
trip_schema = StructType([
    StructField("VendorID",          IntegerType(), True),
    StructField("pickup_datetime",   StringType(),  True),
    StructField("dropoff_datetime",  StringType(),  True),
    StructField("PULocationID",      IntegerType(), True),
    StructField("DOLocationID",      IntegerType(), True),
    StructField("trip_distance",     DoubleType(),  True),
    StructField("fare_amount",       DoubleType(),  True),
    StructField("tip_amount",        DoubleType(),  True),
    StructField("total_amount",      DoubleType(),  True),
    StructField("trip_duration_min", DoubleType(),  True),
    StructField("avg_speed_mph",     DoubleType(),  True),
    StructField("time_of_day",       StringType(),  True),
    StructField("payment_label",     StringType(),  True),
    StructField("event_timestamp",   StringType(),  True),
])

# Kafka 的 value 是二进制，先转字符串再解析 JSON
parsed = (
    raw_stream
    .select(F.from_json(F.col("value").cast("string"), trip_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("pickup_datetime",  F.to_timestamp("pickup_datetime"))
)

print("Parsed schema:")
parsed.printSchema()

Parsed schema:
root
 |-- VendorID: integer (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- avg_speed_mph: double (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- payment_label: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)



## 4. 实时聚合 — 按时段统计行程量和平均票价

In [ ]:
import os

# 用 pickup_datetime（事件时间）做水印，允许最多 2 分钟的迟到数据
# 按上车区域统计每 5 分钟内的叫车量
agg_stream = (
    parsed
    .withWatermark("pickup_datetime", "2 minutes")
    .groupBy(
        F.window("pickup_datetime", "5 minutes"),  # 5 分钟滚动窗口
        "PULocationID"
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "PULocationID",
        "trip_count",
        "avg_fare",
        "avg_duration_min"
    )
)

print("Aggregation stream defined.")

## 5. 启动流处理 — 输出到控制台

In [ ]:
CHECKPOINT_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/streaming/checkpoints/zone-demand")

# outputMode update：每次 trigger 只输出本批次有变化的行，比 complete 更高效
query = (
    agg_stream
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming query started. 等待数据...")
print("现在去 notebook 02 运行 Producer 发送数据！")
print("结果每 10 秒刷新，只显示本批次有新数据的区域。")

query.awaitTermination(timeout=90)
print("\nStreaming 结束。")

## 6. 停止（如需手动停止）

In [6]:
query.stop()
spark.stop()
print("Spark stopped.")

Spark stopped.
